# Module 2 · Light Interception, Solar Time and Daily Integration
**20 minutes: principles 5 · worked example 4 · transfer exercises 8 · discussion 3**

[Course introduction](00_Introduction.ipynb) · [Runnable reference answer](#reference-answer)

**Learning objectives:** distinguish DNI from horizontal irradiance; implement packet-to-PPFD
and trapezoidal-integration expressions; separate instantaneous, peak and cumulative outputs.


In [ ]:
# Run this preparation cell once after starting/restarting the kernel.
# Finds normal, flattened, nested and ZIP-preserved course uploads.
from pathlib import Path
import sys,os,subprocess,hashlib,zipfile,shutil

def locate_canopy_runtime(input_root=None,local_roots=None,work=None,tag=None):
    tag=tag or ('win_amd64' if sys.platform=='win32' else 'manylinux_2_34_x86_64')
    expected=f'canopyteach-1.0.1-cp311-abi3-{tag}.whl'
    pattern=f'canopyteach-1.0.1-cp311-abi3-{tag}*.whl'
    work=Path(work or '/kaggle/working/canopyteach_practical_en')
    candidate_root=Path(input_root or '/kaggle/input')
    wheels=set();helpers=set();archives=set();observed=[];archive_errors=[]

    def consider(p):
        if p.name=='canopy_practical.py':helpers.add(p.resolve())
        elif p.match(pattern):wheels.add(p.resolve())
        elif p.suffix.lower()=='.zip':archives.add(p.resolve())
        if len(observed)<35 and p.is_file():observed.append(str(p))

    if candidate_root.is_dir():
        # Kaggle can add dataset/owner/version directories or flatten uploads.
        for p in candidate_root.rglob('*'):
            if p.is_file():consider(p)
    else:
        roots=local_roots if local_roots is not None else ([] if input_root is not None else [Path.cwd(),*Path.cwd().parents])
        for folder in map(Path,roots):
            if folder==Path(folder.anchor):continue
            for p in [folder/'canopy_practical.py',*folder.glob(pattern),*folder.glob('wheels/'+pattern)]:
                if p.is_file():consider(p)
        if input_root is not None:
            observed.append('Input directory does not exist: '+str(candidate_root))

    # Read only the runtime members of course ZIPs into a writable directory.
    # Never extract arbitrary archive paths or change the input Dataset.
    if not wheels or not helpers:
        for archive in sorted(archives):
            try:
                with zipfile.ZipFile(archive) as z:
                    selected=[n for n in z.namelist() if not n.endswith('/') and
                        (Path(n).name=='canopy_practical.py' or Path(n).match(pattern))]
                    if not selected:continue
                    key=hashlib.sha256(str(archive).encode()).hexdigest()[:12]
                    for i,name in enumerate(selected):
                        info=z.getinfo(name)
                        if info.file_size>30_000_000:raise ValueError('Runtime member exceeds 30 MB: '+name)
                        dest=work/'_runtime'/key/str(i)/Path(name).name
                        dest.parent.mkdir(parents=True,exist_ok=True)
                        dest.write_bytes(z.read(name));consider(dest)
            except (zipfile.BadZipFile,OSError,ValueError) as exc:
                archive_errors.append(str(archive)+': '+str(exc))

    if not wheels or not helpers:
        missing=[]
        if not wheels:missing.append(expected)
        if not helpers:missing.append('canopy_practical.py')
        details='\n'.join(observed) or '(no mounted input files found)'
        raise FileNotFoundError(
            'Missing course runtime: '+', '.join(missing)+'\n'
            'Add the course Dataset as an input to THIS notebook; importing an .ipynb alone does not attach its Dataset.\n'
            'Searched input: '+str(candidate_root)+'\nObserved files (first 35):\n'+details+
            ('\nArchive errors:\n'+'\n'.join(archive_errors) if archive_errors else ''))

    # Prefer a nearby helper, without requiring the wheels/ directory layout.
    pairs=[]
    for wheel in sorted(wheels):
        for helper in sorted(helpers):
            shared=len(os.path.commonpath([str(wheel.parent),str(helper.parent)]).split(os.sep))
            nearby=helper.parent in (wheel.parent,wheel.parent.parent)
            pairs.append((not nearby,-shared,str(wheel),str(helper)))
    _,_,wheel_name,helper_name=min(pairs)
    wheel,helper=Path(wheel_name),Path(helper_name)
    # Browser downloads may add a suffix such as " (1)" to a wheel filename.
    if wheel.name!=expected:
        normalized=work/'_runtime'/'normalized'/expected
        normalized.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(wheel,normalized);wheel=normalized
    return wheel,helper

if __name__=='__main__':
    if sys.version_info<(3,11):raise RuntimeError('This course requires CPython 3.11 or newer.')
    input_override=os.environ.get('CANOPYTEACH_INPUT_ROOT')
    work_override=os.environ.get('CANOPYTEACH_WORK_ROOT')
    on_kaggle=Path('/kaggle/working').is_dir()
    WORK=Path(work_override) if work_override else (
        Path('/kaggle/working/canopyteach_practical_en') if on_kaggle else Path.cwd()/'results/practical_en')
    wheel,helper=locate_canopy_runtime(input_root=input_override,work=WORK)
    PACKAGE_ROOT=helper.parent
    WORK.mkdir(parents=True,exist_ok=True);site=WORK/'_packages';marker=site/'course_wheel.sha256'
    signature=hashlib.sha256(wheel.read_bytes()).hexdigest()
    if not marker.exists() or marker.read_text().strip()!=signature:
        if 'canopyteach' in sys.modules:
            raise RuntimeError('Restart the kernel before installing a different course wheel, then run preparation again.')
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--disable-pip-version-check',
                        '--upgrade','--target',str(site),str(wheel)],check=True)
        marker.write_text(signature)
    sys.path.insert(0,str(site));sys.path.insert(0,str(PACKAGE_ROOT))
    import canopy_practical as cp
    import canopyteach
    if canopyteach.__version__!='1.0.1':raise RuntimeError('Unexpected canopyteach version; restart the kernel.')
    cp.style()
    try:get_ipython().run_line_magic('matplotlib','inline')
    except NameError:pass
    DEFAULT_STEP=int(os.environ.get('CANOPYTEACH_STEP_MINUTES','30'))
    DEFAULT_SPACING=float(os.environ.get('CANOPYTEACH_RAY_SPACING','.005'))
    print('Runtime wheel:',wheel)
    print('Teaching helper:',helper)
    print('Ready. Outputs:',WORK)
    print('Random whole-plant azimuth; central crop; XY periodic; C/OpenMP; synthetic direct light.')


## 1. Principles · 5 min

True solar hour is
$$h_{solar}=h_{clock}+(EoT+4\lambda-60\,UTCoffset)/60.$$
At the reference location/date, clock 12:00 has a correction of about **+2.67176 min**,
giving true solar hour **12.04453**. The solar-time reading is ahead of the clock reading.

Horizontal direct PPFD is $Q_h=DNI\max(s_z,0)$; each packet carries
$e=Q_h\Delta x\Delta y$. Facet PPFD is $Q_i=\sum_{r\to i}e_r/A_i$.
The projection effect is already in the hit density: do not multiply by another cosine.

For hours and a per-second flux, the cumulative quantity in mol/m2 is
```text
sum((rate[:-1]+rate[1:])/2 * diff(hours) * 3600) * 1e-6
```
A peak is a rate, not an integral. Do not multiply by LAI again when integrating a flux
that has already been normalized by ground area.


## 2. Worked example · 4 min

Change `clock_hour` to 9, 12 or 15. This cell creates its own canopy and daily inputs,
then shows central facet light and the photon balance.
DNI is normal to the beam; the balance rates are normalized by central ground area.

Changing the displayed snapshot hour does not change the daily weather program.
The daily calculation follows solar direction at every time step.


In [ ]:
import matplotlib.pyplot as plt
import canopy_practical as cp
cp.style()

clock_hour=12.  # EDIT: 9, 12 or 15; geometry and light model stay fixed.
step_minutes=DEFAULT_STEP
ray_spacing=DEFAULT_SPACING
sp,light,run,fig=cp.light_example(clock_hour,step_minutes,ray_spacing)
print('Solar-hour correction (min):',sp['correction_minutes'])
print('True solar hour:',sp['solar_hour'])
print('Leaf/stem/escape fractions:',light.fractions)
cp.photon_table(run);plt.show()


## 3. Programming exercises 2A–2B · 6 min

**2A · Convert packets to PPFD.** A 0.002 m2 facet receives 240 packets, each carrying
0.01 umol photons/s. Complete `student_ppfd`: the self-check should return **1200**.
Do not apply an additional leaf-normal cosine.

**2B · Integrate a flux.** Complete `student_daily_integral` using the supplied time
differences and mean interval rates. Include both unit conversions.
A constant 10 umol CO2/(m2 ground s) for 0.5 h should give **0.018 mol/m2 ground**.
Then apply the same function to the actual central canopy's leaf-photon rate.
The default result is about **41.881273 mol photons/m2 ground/day**.
If you change sampling, compare against your actual model output.

The verified photon table and curve remain available while the TODOs are unfinished.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import canopy_practical as cp

def student_ppfd(hit_rays,energy_per_ray,face_area):
    return None  # TODO 2A: energy rate reaching face / face area.

def student_daily_integral(hours,rate):
    hours=np.asarray(hours);rate=np.asarray(rate)
    dt_hours=np.diff(hours)
    average_rate=(rate[:-1]+rate[1:])/2
    return None  # TODO 2B: sum trapezoids; hours -> seconds; umol -> mol.

q=student_ppfd(240,.01,.002)
c=student_daily_integral([0,.5],[10,10])
if q is None or c is None:
    print('Unfinished: complete both return expressions. Checks: Q=1200 and C=0.018.')
else:
    np.testing.assert_allclose(q,1200.)
    np.testing.assert_allclose(c,.018)
    print('Both analytic self-checks passed.')
# The actual model output remains available while the analysis TODOs are unfinished.
run=cp.day(step_minutes=DEFAULT_STEP,ray_spacing=DEFAULT_SPACING)
cp.photon_table(run)
value=student_daily_integral(run.hours,cp.photon_rates(run))
if value is not None:
    expected=run.summary['daily_leaf_intercepted_mol_photons_m2_ground']
    np.testing.assert_allclose(value,expected)
    print('Student daily leaf integral:',value,'mol photons/m2 ground')
fig,ax=plt.subplots(figsize=(7,3));ax.plot(run.hours,cp.photon_rates(run),color='#247c6b')
ax.set(xlabel='Clock hour',ylabel='Leaf photons / umol m-2 ground s-1');fig.tight_layout();plt.show()


## 4. Independent snapshot experiment 2C · 2 min

After preparation, this cell runs independently. Change `clock_hour` and compare true solar
time, horizontal direct light and the three fractions.
Explain why horizontal irradiance changes even though DNI is fixed. Can a larger noon PPFD
alone establish a larger daily integral?


In [ ]:
from canopyteach import CanopyLight,solar_position
from canopyteach.simulation import prepare_canopy
import matplotlib.pyplot as plt
import canopy_practical as cp
cp.style()

clock_hour=15.  # EDIT: a different time; no previous experiment variables are needed.
mesh=prepare_canopy();sp=solar_position('2026-06-21',clock_hour,31,121,8)
light=CanopyLight(mesh,ray_spacing=DEFAULT_SPACING,threads=4).trace(sp['sun'],1500.)
print('Clock / solar hour:',clock_hour,sp['solar_hour'])
print('Direct-normal=1500; horizontal=',1500*max(0,sp['sun'][2]))
print('Fractions:',light.fractions,'sum=',sum(light.fractions.values()))
fig=plt.figure(figsize=(6,4.8));ax=fig.add_subplot(projection='3d')
cp.draw_mesh(ax,mesh.mesh,light.total,title=f'Central crop: clock {clock_hour:g} h',cmap='YlOrRd',vmax=1800)
plt.show()


<a id="reference-answer"></a>
## 5. Reference answers and discussion · 3 min

**2A:** `N*e/area`. **2B:** sum trapezoids, multiply by 3600 and 1e-6.
The default daily balance is approximately
41.881273 leaf + 0.860242 stem + 4.072990 escape = 46.814505 incident photons,
all in mol/m2 ground.

**2C:** solar elevation changes `s_z`. DNI and horizontal direct PPFD differ.
Do not multiply noon flux by 24 h, integrate hours as seconds, or treat a peak as a total.
Run the independent reference below to check your integration without replacing your functions.
Continue to [Module 3](03_Photosynthesis_and_Design.ipynb).


In [ ]:
def _reference_light(step,spacing):
    import numpy as np
    import canopy_practical as cp
    def ppfd(n,e,area):return n*e/area
    def integral(hours,rate):
        h=np.asarray(hours);v=np.asarray(rate)
        return np.sum((v[:-1]+v[1:])*.5*np.diff(h)*3600)*1e-6
    print('Analytic PPFD:',ppfd(240,.01,.002))
    print('Analytic 30-minute C:',integral([0,.5],[10,10]))
    r=cp.day(step_minutes=step,ray_spacing=spacing)
    value=integral(r.hours,cp.photon_rates(r))
    np.testing.assert_allclose(value,r.summary['daily_leaf_intercepted_mol_photons_m2_ground'])
    print('Daily leaf photons:',value);cp.photon_table(r)
_reference_light(DEFAULT_STEP,DEFAULT_SPACING)
